# Fig 6c: model fLoc selectivity by stream
Analyze contrast t-values for units assigned to different streams.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import h5py

from itertools import combinations
from scipy.stats import ttest_rel

# Use Helvetica when exporting vector figures.
mpl.rcParams['pdf.use14corefonts'] = True
mpl.rcParams['ps.useafm'] = True
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Helvetica']

from spacestream.core.constants import SUBJECTS, CORE_ROI_NAMES
from spacestream.core.paths import NSDDATA_PATH, RESULTS_PATH
from spacestream.utils.get_utils import get_mapping

In [ ]:
# Parameters
seeds = [0, 1, 2, 3, 4]
hemis = ['lh', 'rh']
spatial_weights = [0.25, 0.5]
checkpoint = 'checkpoint0'
sel_meta_path = RESULTS_PATH + 'analyses/selectivity/20230811_sel_meta.csv'
sel_arrays_path = RESULTS_PATH + 'analyses/selectivity/20230811_sel_arrays.npz'


In [ ]:
# Load selectivity metadata + arrays
sel = np.load(sel_arrays_path)
meta = pd.read_csv(sel_meta_path)
meta_t = meta[(meta['sel_fn'] == 't-stat') & (meta['spatial_weight'].isin(spatial_weights))]
contrasts = list(meta_t['contrast'].unique())
contrasts = ['Places', 'Bodies', 'Faces']
print('Contrasts:', contrasts)
print('Spatial weights:', spatial_weights)


In [ ]:
def get_tvals_for_seed_contrast(seed, contrast):
    row = meta_t[(meta_t['seed'] == seed) & (meta_t['contrast'] == contrast)]
    if len(row) != 1:
        raise ValueError(f'Expected 1 row for seed={seed}, contrast={contrast}, got {len(row)}')
    idx = int(row['Unnamed: 0'].iloc[0])
    return sel['arrays'][idx]


In [ ]:
# Build per-subject summary of mean t-values by stream and contrast
rows = []

for spatial_weight in spatial_weights:
    meta_sw = meta_t[meta_t['spatial_weight'] == spatial_weight]

    for seed in seeds:
        tvals_by_contrast = {}
        for contrast in contrasts:
            row = meta_sw[(meta_sw['seed'] == seed) & (meta_sw['contrast'] == contrast)]
            if len(row) != 1:
                raise ValueError(
                    f'Expected 1 row for sw={spatial_weight}, seed={seed}, contrast={contrast}, got {len(row)}'
                )
            idx = int(row['Unnamed: 0'].iloc[0])
            tvals_by_contrast[contrast] = sel['arrays'][idx]

        for hemi in hemis:
            for subj in SUBJECTS:
                subj_name = 'subj' + subj
                mapping = get_mapping(
                    subj_name, 'unit2voxel', str(spatial_weight), seed, 0, hemi, checkpoint
                )

                for stream_idx, stream in enumerate(CORE_ROI_NAMES):
                    stream_units = np.where(mapping['winning_roi'] == 5 + stream_idx)[0]

                    for contrast in contrasts:
                        tvals = tvals_by_contrast[contrast][stream_units]
                        rows.append({
                            'subject': subj,
                            'hemi': hemi,
                            'seed': seed,
                            'spatial_weight': spatial_weight,
                            'stream': stream,
                            'contrast': contrast,
                            'mean_t': np.mean(tvals),
                            'n_units': len(stream_units),
                        })

df = pd.DataFrame(rows)
df.head()


In [ ]:
# Average across seeds and hemis per subject
df_avg = (
    df.groupby(['subject', 'stream', 'contrast', 'spatial_weight'])['mean_t']
    .mean()
    .reset_index()
)
df_avg.head()


In [ ]:
# Per-subject/hemi/spatial-weight averages across seeds (for dots)
df_avg_hemi = (
    df.groupby(['subject', 'hemi', 'stream', 'contrast', 'spatial_weight'])['mean_t']
    .mean()
    .reset_index()
)
df_avg_hemi.head()


In [ ]:
# Pairwise stream comparisons (paired t-test across subjects)
results = []

for spatial_weight in spatial_weights:
    df_sw = df_avg[df_avg['spatial_weight'] == spatial_weight]

    for contrast in contrasts:
        pivot = df_sw[df_sw['contrast'] == contrast].pivot(
            index='subject', columns='stream', values='mean_t'
        )

        for a, b in combinations(CORE_ROI_NAMES, 2):
            paired = pivot[[a, b]].dropna()
            if len(paired) == 0:
                continue
            stat, pval = ttest_rel(paired[a], paired[b])
            results.append({
                'spatial_weight': spatial_weight,
                'contrast': contrast,
                'stream_a': a,
                'stream_b': b,
                'n_subjects': len(paired),
                't_stat': stat,
                'p_value': pval,
            })

results_df = pd.DataFrame(results)
results_df['p_value_bonf'] = (results_df['p_value'] * 9).clip(upper=1.0)
results_df = results_df.sort_values(['spatial_weight', 'contrast', 'p_value_bonf'])
results_df


In [ ]:
# Count units with t > threshold per stream/contrast
t_threshold = 3
count_rows = []

for spatial_weight in spatial_weights:
    meta_sw = meta_t[meta_t['spatial_weight'] == spatial_weight]

    for seed in seeds:
        tvals_by_contrast = {}
        for contrast in contrasts:
            row = meta_sw[(meta_sw['seed'] == seed) & (meta_sw['contrast'] == contrast)]
            if len(row) != 1:
                raise ValueError(
                    f'Expected 1 row for sw={spatial_weight}, seed={seed}, contrast={contrast}, got {len(row)}'
                )
            idx = int(row['Unnamed: 0'].iloc[0])
            tvals_by_contrast[contrast] = sel['arrays'][idx]

        for hemi in hemis:
            for subj in SUBJECTS:
                subj_name = 'subj' + subj
                mapping = get_mapping(
                    subj_name, 'unit2voxel', str(spatial_weight), seed, 0, hemi, checkpoint
                )

                for stream_idx, stream in enumerate(CORE_ROI_NAMES):
                    stream_units = np.where(mapping['winning_roi'] == 5 + stream_idx)[0]

                    for contrast in contrasts:
                        tvals = tvals_by_contrast[contrast][stream_units]
                        count_rows.append({
                            'subject': subj,
                            'hemi': hemi,
                            'seed': seed,
                            'spatial_weight': spatial_weight,
                            'stream': stream,
                            'contrast': contrast,
                            'n_units': len(stream_units),
                            'n_t_above': int(np.sum(tvals > t_threshold)),
                            'frac_t_above': float(np.mean(tvals > t_threshold)) if len(stream_units) else np.nan,
                        })

df_count = pd.DataFrame(count_rows)
df_count.head()


In [ ]:
# Average counts across seeds and hemis per subject
df_count_avg = (
    df_count.groupby(['subject', 'stream', 'contrast', 'spatial_weight'])[['n_units', 'n_t_above', 'frac_t_above']]
    .mean()
    .reset_index()
)
df_count_avg.head()


In [ ]:
# Per-subject/hemi/spatial-weight averages across seeds (for dots)
df_count_avg_hemi = (
    df_count.groupby(['subject', 'hemi', 'stream', 'contrast', 'spatial_weight'])[['n_units', 'n_t_above', 'frac_t_above']]
    .mean()
    .reset_index()
)
df_count_avg_hemi.head()


In [ ]:
sns.set_theme(style='ticks', context='talk')

plot_order = contrasts
hue_order = CORE_ROI_NAMES[::-1]  # ['Ventral', 'Lateral', 'Dorsal']

ROI_COLORS = ['#377E2C', '#1A1AAC', '#8C1A4C']
stream_palette = {
    'Ventral': ROI_COLORS[2],
    'Lateral': ROI_COLORS[1],
    'Dorsal': ROI_COLORS[0],
}

# Final paper panel: match Fig 1b by computing bars/error bars from one value per subject/stream/contrast.
# df_count_avg already averages across seeds and hemispheres per subject/spatial weight;
# this additional average prevents spatial weights from acting as independent samples.
df_count_subject = (
    df_count_avg[df_count_avg['contrast'].isin(plot_order)]
    .groupby(['subject', 'stream', 'contrast'], as_index=False)['frac_t_above']
    .mean()
)

# Overlay dots at hemisphere-level for visibility, averaged across seeds and spatial weights.
# These dots show both lh/rh but are not the error-bar sample.
df_count_hemi = (
    df_count_avg_hemi[df_count_avg_hemi['contrast'].isin(plot_order)]
    .groupby(['subject', 'hemi', 'stream', 'contrast'], as_index=False)['frac_t_above']
    .mean()
)

fig, ax = plt.subplots(figsize=(6, 6))

sns.barplot(
    data=df_count_subject,
    x='contrast',
    y='frac_t_above',
    hue='stream',
    order=plot_order,
    hue_order=hue_order,
    palette=stream_palette,
    ci='sd',        # older seaborn API: mean +/- 1 SD
    capsize=0.08,
    errwidth=1.5,
    edgecolor="white",
    linewidth=4,
    ax=ax,
)

markers = {'lh': 'o', 'rh': '^'}
for hemi, marker in markers.items():
    sns.stripplot(
        data=df_count_hemi[df_count_hemi['hemi'] == hemi],
        x='contrast',
        y='frac_t_above',
        hue='stream',
        order=plot_order,
        hue_order=hue_order,
        dodge=True,
        jitter=0.15,
        marker=marker,
        size=4.5,
        edgecolor='white',
        linewidth=0.5,
        palette=stream_palette,
        ax=ax,
    )

handles, labels = ax.get_legend_handles_labels()
stream_handles = {}
for h, lab in zip(handles, labels):
    if lab in hue_order and lab not in stream_handles:
        stream_handles[lab] = h
ax.legend(stream_handles.values(), stream_handles.keys(), frameon=False, title='Stream')

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.set_xlabel('Contrast')
ax.set_ylabel(f'Fraction t > {t_threshold}')
plt.tight_layout()
# Best for papers: vector format (infinitely scalable)
fig = ax.figure
fig.savefig("F06_C_floc_tvalues_by_stream.pdf", bbox_inches="tight")   # or .svg/.eps

ax.grid(False)
ax.tick_params(colors="black", labelcolor="black")
ax.spines["left"].set_color("black")
ax.spines["bottom"].set_color("black")
plt.show()


In [ ]:
# Pairwise stream comparisons on fraction t > threshold (paired t-test across subjects)
# Average across hemispheres and spatial weights per subject
count_results = []

df_sw_hemi_avg = (
    df_count_avg_hemi.groupby(['subject', 'stream', 'contrast'])['frac_t_above']
    .mean()
    .reset_index()
)

for contrast in contrasts:
    pivot = df_sw_hemi_avg[df_sw_hemi_avg['contrast'] == contrast].pivot(
        index='subject', columns='stream', values='frac_t_above'
    )

    for a, b in combinations(CORE_ROI_NAMES, 2):
        paired = pivot[[a, b]].dropna()
        if len(paired) == 0:
            continue
        stat, pval = ttest_rel(paired[a], paired[b])
        count_results.append({
            'contrast': contrast,
            'stream_a': a,
            'stream_b': b,
            'n_subjects': len(paired),
            't_stat': stat,
            'p_value': pval,
        })

count_results_df = pd.DataFrame(count_results)
count_results_df['p_value_bonf'] = (count_results_df['p_value'] * 9).clip(upper=1.0) #9 because running 9 tests
count_results_df = count_results_df.sort_values(['contrast', 'p_value_bonf'])
count_results_df
